### Step A — Create training examples  

We need pairs of: Instruction (human question) Output (correct Cypher query)

Example:
- Instruction: “List actors in Casino”
- Output: MATCH (m:Movie {title:'Casino'})<-[:ACTED_IN]-(p:Person) RETURN p.name;
Why: The model learns the pattern “question → Cypher”.

### Step B — Convert each example into ONE training text
- Models learn by predicting next tokens. So we join instruction and answer into a single text:

### Instruction:
List actors in the movie Casino.

### Response:
- MATCH (m:Movie {title:'Casino'})<-[:ACTED_IN]-(p:Person) RETURN p.name;
- Why: The model learns to “complete” the response after seeing instruction.

### Step C — Tokenize (convert words → numbers)
- Models don’t understand letters. They understand token IDs (numbers).
- Tokenizer turns the text into tokens.
Why: Training happens on token IDs.

### Step D — Load a base model (small, for learning)
- We’ll use a small model (distilgpt2) so it runs fast.
- Why: First make pipeline work. Later you can switch to a bigger chat model.

### Step E — Add LoRA adapters (the key idea)
- LoRA: freezes the big model adds a small trainable “patch” to some layers
- Why: cheap training, low memory.

Important LoRA knobs:
r (rank): adapter capacity (how much it can learn)
alpha: strength of LoRA update
dropout: prevents memorizing tiny dataset
target_modules: where to attach LoRA (for GPT2 it’s c_attn)

### Step F — Train only the adapters
We run a small training loop:
- send tokenized examples
- model predicts next tokens
- updates LoRA adapter weights only
- Why: base model stays intact.

### Step G — Save the adapter
We save only LoRA adapter (small files).
Later we load:
base model
adapter on top

### Step H — Test
Give a new question and see if it generates valid Cypher

1) Install libraries

In [1]:
!pip -q install -U transformers datasets peft accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 84.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.7 MB/s eta 0:00:00:00:0100:01


2) Make your tiny Neo4j training data

In [2]:
from datasets import Dataset

data = [
    {"instruction": "How many actors are there?",
     "output": "MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a);"},
    {"instruction": "Which actors played in the movie Casino?",
     "output": "MATCH (m:Movie {title:'Casino'})<-[:ACTED_IN]-(a:Person) RETURN a.name;"},
    {"instruction": "How many movies has Tom Hanks acted in?",
     "output": "MATCH (a:Person {name:'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN count(m);"},
]

ds = Dataset.from_list(data)
ds

Dataset({
    features: ['instruction', 'output'],
    num_rows: 3
})

3) Choose model (small first)

4) Load tokenizer + create training text format

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [5]:
def format_row(row):
    text = f"### Instruction:\n{row['instruction']}\n\n### Response:\n{row['output']}"
    return {"text": text}

ds_formatted = ds.map(format_row)
ds_formatted[0]["text"]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

'### Instruction:\nHow many actors are there?\n\n### Response:\nMATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a);'

In [ ]:
# 5) Tokenize (text → numbers)
max_len = 256

# It takes text and converts it into numbers.
# "MATCH (m:Movie {title:'Casino'}) RETURN a.name;" 
""" {
  "input_ids": [50256, 1234, 5678, ...],
  "attention_mask": [1, 1, 1, ...]
}{
  "input_ids": [50256, 1234, 5678, ...],
  "attention_mask": [1, 1, 1, ...]
} """ 


def tokenize_fn(batch): 
    return tokenizer(batch["text"], truncation=True, max_length=max_len)

tokenized = ds_formatted.map(tokenize_fn, batched=True, remove_columns=ds_formatted.column_names)
tokenized

# output tokenized[0] 
""" {
  "input_ids": [15496, 1077, 2307, ...],
  "attention_mask": [1, 1, 1, ...]
} """

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 3
})

Why this step is CRITICAL for LoRA training

LoRA training still uses normal language model training.

The model learns:

“Given token 1, predict token 2… then token 3…”

So we must:

Combine instruction + answer into one sequence

Convert it to token IDs

Feed only token IDs to the trainer

This line does exactly that. 

In [9]:
model_id = "distilgpt2"

In [16]:
# 6) Load base model 
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_id)

In [18]:
# 7) Attach LoRA adapters 
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn"],  # correct for GPT2
    bias="none"
)

model = get_peft_model(model, lora_config)
print(type(model))
model.print_trainable_parameters()

<class 'peft.peft_model.PeftModelForCausalLM'>
trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [19]:
# 8) Train (simple Trainer)

#TrainingArguments → configuration (knobs)
#Trainer → training loop (forward → loss → backward → update)
#DataCollatorForLanguageModeling → prepares batches correctly
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling


# examples have different lengths.pads them to the same length.creates labels for training
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
# mlm=True → BERT-style “fill the blank”
# mlm=False → GPT-style “predict next token”
# collator produces id → input_ids, attention_mask, labels 
# labels are what the model tries to predict 

training_args = TrainingArguments(
    output_dir="lora_demo_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=10,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

trainer.train()

""" 
1️⃣ Take batch from dataset
2️⃣ Pad sequences → same length
3️⃣ Feed input_ids to model
4️⃣ Model predicts next token
5️⃣ Compare prediction vs labels
6️⃣ Compute loss
7️⃣ Backpropagate loss
8️⃣ Update only LoRA weights
9️⃣ Repeat until dataset is done
🔁 Repeat for next epoch"""


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,5.475500
2,5.420100
3,5.352700
4,5.439500
5,5.320200
6,5.244400
7,5.381400
8,5.428600
9,5.326200
10,5.323100


' \n1️⃣ Take batch from dataset\n2️⃣ Pad sequences → same length\n3️⃣ Feed input_ids to model\n4️⃣ Model predicts next token\n5️⃣ Compare prediction vs labels\n6️⃣ Compute loss\n7️⃣ Backpropagate loss\n8️⃣ Update only LoRA weights\n9️⃣ Repeat until dataset is done\n🔁 Repeat for next epoch'

In [20]:
#v Step 9 — Save the LoRA adapter
model.save_pretrained("lora_adapter_v1")
tokenizer.save_pretrained("lora_adapter_v1")
import os
os.listdir("lora_adapter_v1")

['README.md',
 'adapter_model.safetensors',
 'adapter_config.json',
 'tokenizer.json',
 'merges.txt',
 'tokenizer_config.json',
 'vocab.json',
 'special_tokens_map.json']

In [ ]:
# Step 10 — Test the fine-tuned model 
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(model_id)
ft = PeftModel.from_pretrained(base, "lora_adapter_v1")
ft.eval() # We are not training. Be stable and deterministic. 

prompt = "### Instruction:\nList actors in the movie Casino.\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    out = ft.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

""" 
1️⃣ reads your prompt tokens
2️⃣ predicts the next token
3️⃣ appends it
4️⃣ repeats until:

60 tokens generated, or

end-of-sentence token """

print(tokenizer.decode(out[0], skip_special_tokens=True)) 
# [15496, 7, 182, ...] to MATCH (m:Movie ...) RETURN ... 
# skip_special_tokens=True removes tokens like: <eos> , <pad>


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


### Instruction:
List actors in the movie Casino.

### Response:
The movie Casino is a movie that is based on the movie Casino.
### Response:
The movie Casino is a movie that is based on the movie Casino.
### Response:
The movie Casino is a movie that is based on the movie Casino.
### Response:
The movie Casino


In [15]:
print(type(model))
model.print_trainable_parameters()

<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>


AttributeError: 'GPT2LMHeadModel' object has no attribute 'print_trainable_parameters'